# CI/CD for Machine Learning

Notebook for **Chapter 9** of the *mlops* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/mlops/ch09.html

Tip: change a number, predict the output, then run the cell again.

## 9.1 Continuous Integration for ML

## 9.2 Model Tests: Testing Behaviour, Not Just Accuracy

In [ ]:
def test_directional_debt():   # more debt must not LOWER default risk
    assert p([50, 0.8]) > p([50, 0.2])
def test_directional_income():  # more income must not RAISE default risk
    assert p([80, 0.5]) < p([30, 0.5])
def test_invariance_noise():   # tiny change of income barely moves the score
    assert abs(p([50.0, 0.5]) - p([50.1, 0.5])) < 0.01
def test_range():
    for row in [[0, 0], [500, 1], [50, 0.5]]: assert 0 <= p(row) <= 1
def test_performance_floor():
    acc = model.score((X - mu) / sd, y); assert acc > 0.60, acc
def test_broken_example():
    assert p([50, 0.2]) > p([50, 0.8])   # deliberately wrong test

## 9.3 Promotion Gates and Continuous Delivery

In [ ]:
def gate(champ, cand, p95_ms, min_gain=0.005, max_p95=100):
    reasons = []
    if cand["auc"] < champ["auc"] + min_gain:
        reasons.append(f"AUC gain {cand['auc']-champ['auc']:+.3f} < {min_gain}")
    if p95_ms > max_p95:
        reasons.append(f"p95 {p95_ms} ms > {max_p95} ms")
    return reasons          # empty list means PASS

## 9.4 Continuous Training and MLOps Maturity

In [ ]:
def should_retrain(days_since, psi, new_rows, live_acc, base_acc):
    reasons = []
    if days_since >= 30: reasons.append(f"schedule ({days_since} days)")
    if psi > 0.2: reasons.append(f"drift (PSI {psi})")
    if new_rows >= 50_000: reasons.append(f"data volume ({new_rows} new rows)")
    if live_acc is not None and base_acc - live_acc > 0.03:
        reasons.append(f"performance drop ({base_acc-live_acc:.3f})")
    return reasons

(10, 0.04, 8000, 0.84, 0.85) -> no retrain
(31, 0.05, 9000, 0.85, 0.85) -> ['schedule (31 days)']
(12, 0.31, 7000, None, 0.85) -> ['drift (PSI 0.31)']
(9, 0.08, 61000, 0.8, 0.85) -> ['data volume (61000 new rows)', 'performance drop (0.050)']


## 9.5 An End-to-End Pipeline: Contracts, Gates, Promotion and GitOps

Section 9.1 to 9.4 gave you the parts: tests, gates, promotion and triggers. Here you connect them into one pipeline you could copy into a real repository. The Python is real and ran here (with scikit-learn, pandas and PyYAML). The GitHub Actions and Argo CD files cannot run here (they need GitHub and a cluster), so they are labelled not run here, but I parsed the workflow file with a YAML parser to check that its structure is valid.

In [ ]:
import pandas as pd, numpy as np
# A data contract: the rules a new batch must satisfy before training or scoring.
contract = {
    "income":      {"type": "float", "min": 0, "max": 1_000_000, "max_null": 0.02},
    "debt_ratio":  {"type": "float", "min": 0, "max": 5,         "max_null": 0.00},
    "region":      {"type": "str",   "allowed": {"north", "south", "east", "west"}, "max_null": 0.00},
}
def check(df, contract):
    problems = []
    for col, rule in contract.items():
        if col not in df:
            problems.append(f"{col}: column missing"); continue
        s = df[col]
        null = s.isna().mean()
        if null > rule["max_null"]:
            problems.append(f"{col}: null rate {null:.1%} above {rule['max_null']:.1%}")
        s = s.dropna()
        if "min" in rule and (s.lt(rule["min"]).any() or s.gt(rule["max"]).any()):
            problems.append(f"{col}: {int((s.lt(rule['min']) | s.gt(rule['max'])).sum())} values outside [{rule['min']}, {rule['max']}]")
        if "allowed" in rule:
            bad = set(s) - rule["allowed"]
            if bad: problems.append(f"{col}: unexpected values {sorted(bad)}")
    return problems

good = pd.DataFrame({"income": [40e3, 52e3, 47e3, 61e3] * 50, "debt_ratio": [0.3, 0.5, 0.2, 0.4] * 50,
                     "region": ["north", "south", "east", "west"] * 50})
bad = good.copy()
bad.loc[:9, "income"] = np.nan                  # a broken join upstream
bad.loc[10, "debt_ratio"] = 17.0                # a unit error
bad.loc[11, "region"] = "NORTH"                 # a case change
print("good batch:", check(good, contract) or "OK")
for p in check(bad, contract): print("bad batch:", p)

good batch: OK
bad batch: income: null rate 5.0% above 2.0%
bad batch: debt_ratio: 1 values outside [0, 5]
bad batch: region: unexpected values ['NORTH']


In [ ]:
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(7)
n = 6000
income = rng.normal(55, 15, n).clip(10, 120)
debt = rng.uniform(0, 1.5, n)
young = rng.random(n) < 0.3                       # a slice we must not fail
z = -1.0 + 2.2 * debt - 0.03 * (income - 55)
y = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)
X = np.column_stack([income, debt])
tr, te = slice(0, 4500), slice(4500, None)

def gate_report(model):
    model.fit(X[tr], y[tr])
    p = model.predict_proba(X[te])[:, 1]
    yt = young[te]
    r = {"auc_overall": roc_auc_score(y[te], p)}
    r["slice_gap"] = abs(roc_auc_score(y[te][yt], p[yt]) - roc_auc_score(y[te][~yt], p[~yt]))
    grid = np.column_stack([np.full(50, 55.0), np.linspace(0, 1.5, 50)])
    pr = model.predict_proba(grid)[:, 1]            # raise debt, hold income fixed
    r["monotone_violation"] = float(max(0, (pr[:-1] - pr[1:]).max()))
    return r

def verdict(r):
    gates = [("overall AUC >= 0.70", r["auc_overall"] >= 0.70),
             ("slice AUC gap <= 0.05", r["slice_gap"] <= 0.05),
             ("debt monotone (violation <= 0.02)", r["monotone_violation"] <= 0.02)]
    for name, ok in gates: print("  " + ("PASS " if ok else "FAIL ") + name)
    return all(ok for _, ok in gates)

for label, m in (("free model", HistGradientBoostingClassifier(random_state=0)),
                 ("monotone model", HistGradientBoostingClassifier(random_state=0, monotonic_cst=[-1, 1]))):
    r = gate_report(m)
    print(label, {k: round(v, 3) for k, v in r.items()})
    print("  exit code", 0 if verdict(r) else 1)

free model {'auc_overall': 0.735, 'slice_gap': 0.014, 'monotone_violation': 0.21}
  PASS overall AUC >= 0.70
  PASS slice AUC gap <= 0.05
  FAIL debt monotone (violation <= 0.02)
  exit code 1
monotone model {'auc_overall': 0.755, 'slice_gap': 0.003, 'monotone_violation': 0.0}
  PASS overall AUC >= 0.70
  PASS slice AUC gap <= 0.05
  PASS debt monotone (violation <= 0.02)
  exit code 0


In [ ]:
import yaml
WORKFLOW = """
name: model-ci
on:
  pull_request:
    paths: ["src/**", "tests/**", "configs/**", "dvc.lock"]
  workflow_dispatch:

jobs:
  fast:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: {python-version: "3.11", cache: pip}
      - run: pip install -r requirements.lock
      - run: ruff check src tests
      - run: pytest tests/unit tests/data -q
  model-gate:
    needs: fast
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: {python-version: "3.11", cache: pip}
      - run: pip install -r requirements.lock
      - run: dvc pull data/sample.csv
      - run: python -m src.train --config configs/ci.yaml --out out/
      - run: python -m src.gate out/report.json --config configs/gates.yaml
      - uses: actions/upload-artifact@v4
        if: always()
        with: {name: gate-report, path: out/report.json}
  promote:
    needs: model-gate
    if: github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest
    environment: production        # a human approves here
    steps:
      - uses: actions/checkout@v4
      - run: python -m src.promote --model churn --alias champion --from-run "${{ github.sha }}"
"""
wf = yaml.safe_load(WORKFLOW)
print("triggers:", list(wf[True].keys()))        # YAML reads the key "on" as the boolean True
for name, job in wf["jobs"].items():
    print(f"{name:11s} needs={job.get('needs')} steps={len(job['steps'])} env={job.get('environment')}")

triggers: ['pull_request', 'workflow_dispatch']
fast        needs=None steps=5 env=None
model-gate  needs=fast steps=7 env=None
promote     needs=model-gate steps=2 env=production


In [ ]:
# A registry with aliases: promotion moves a pointer, it does not copy a model.
registry = {"churn": {"versions": {1: "run_a1", 2: "run_b7", 3: "run_c4"}, "aliases": {"champion": 2}}}

def promote(name, version, alias="champion", gates_passed=False):
    if not gates_passed:
        raise RuntimeError("refused: gates did not pass")
    old = registry[name]["aliases"].get(alias)
    registry[name]["aliases"][alias] = version
    return old

print("serving loads version:", registry["churn"]["aliases"]["champion"])
try:
    promote("churn", 3, gates_passed=False)
except RuntimeError as e:
    print(e)
old = promote("churn", 3, gates_passed=True)
print("promoted 3, previous was", old, "->", registry["churn"]["aliases"])
promote("churn", old, gates_passed=True)             # rollback is the same one-line move
print("rolled back ->", registry["churn"]["aliases"])

serving loads version: 2
refused: gates did not pass
promoted 3, previous was 2 -> {'champion': 3}
rolled back -> {'champion': 2}


## Chapter recap

## Practice set

**1.** (Easy) Which exit code means a CI step failed?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/mlops/ch09.html